In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
)

# Create the widget once; retain its current value when rerunning the cell.
if "batch_id" not in dbutils.widgets.getAll():
    dbutils.widgets.text(
        "batch_id",
        "full-sample-20261010-01",
        "Bronze batch ID",
    )

batch_id = dbutils.widgets.get("batch_id").strip()
if not batch_id:
    raise ValueError("batch_id is required")

bronze_table = "workspace.tmdb_lakehouse.bronze_movie_records"
bronze_batch_df = (
    spark.table(bronze_table)
    .filter(F.col("batch_id") == batch_id)
)

bronze_count = bronze_batch_df.count()
if bronze_count == 0:
    raise ValueError(f"No Bronze records found for batch_id={batch_id}")

print(f"Bronze rows in selected batch: {bronze_count}")
bronze_batch_df.select(
    "batch_id",
    "load_type",
    "source_extracted_at",
).distinct().show(truncate=False)

child_movie_schema = StructType([
    StructField("id", LongType(), True),

    StructField("genres", ArrayType(StructType([
        StructField("id", LongType(), True),
        StructField("name", StringType(), True),
    ])), True),

    StructField("production_companies", ArrayType(StructType([
        StructField("id", LongType(), True),
        StructField("name", StringType(), True),
        StructField("origin_country", StringType(), True),
    ])), True),

    StructField("credits", StructType([
        StructField("cast", ArrayType(StructType([
            StructField("id", LongType(), True),
            StructField("credit_id", StringType(), True),
            StructField("name", StringType(), True),
            StructField("character", StringType(), True),
            StructField("order", IntegerType(), True),
        ])), True),
    ]), True),

    StructField("keywords", StructType([
        StructField("keywords", ArrayType(StructType([
            StructField("id", LongType(), True),
            StructField("name", StringType(), True),
        ])), True),
    ]), True),
])

parsed_child_movies_df = (
    bronze_batch_df
    .select(
        "movie_id",
        "batch_id",
        "load_type",
        "source_file",
        "source_extracted_at",
        F.from_json("raw_payload", child_movie_schema).alias("movie"),
    )
    .select(
        "movie_id",
        "batch_id",
        "load_type",
        "source_file",
        "source_extracted_at",
        "movie.*",
    )
)

parsed_count = parsed_child_movies_df.count()
bad_ids = parsed_child_movies_df.filter(
    F.col("id").isNull() | (F.col("id") != F.col("movie_id"))
).count()

print(f"Parsed movie rows: {parsed_count}")
print(f"Missing or mismatched parsed IDs: {bad_ids}")
parsed_child_movies_df.select(
    "movie_id",
    "id",
    F.size("genres").alias("genre_count"),
    F.size("production_companies").alias("studio_count"),
    F.size("credits.cast").alias("cast_count"),
    F.size("keywords.keywords").alias("keyword_count"),
).show(10, truncate=False)

if bad_ids:
    raise ValueError(
        f"{bad_ids} Bronze records have a missing or mismatched movie ID"
    )

In [0]:
from pyspark.sql import functions as F

# A missing collection is excluded; a present empty array remains a snapshot candidate.
genre_snapshot_candidates_df = (
    parsed_child_movies_df
    .filter(F.col("genres").isNotNull())
    .select(
        "movie_id",
        "source_extracted_at",
        F.col("batch_id").alias("source_batch_id"),
        F.size("genres").alias("genre_count"),
    )
)

genre_candidate_duplicates = (
    genre_snapshot_candidates_df
    .groupBy("movie_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
if genre_candidate_duplicates:
    raise ValueError(
        f"Found {genre_candidate_duplicates} duplicate movie snapshot candidates"
    )

silver_genres_source_df = (
    parsed_child_movies_df
    .filter(F.col("genres").isNotNull())
    .select(
        "movie_id",
        "batch_id",
        "source_file",
        "source_extracted_at",
        F.explode("genres").alias("genre"),
    )
    .select(
        "movie_id",
        F.col("genre.id").alias("genre_id"),
        F.col("genre.name").alias("genre_name"),
        F.col("batch_id").alias("source_batch_id"),
        "source_file",
        "source_extracted_at",
    )
)

invalid_genres_df = silver_genres_source_df.filter(
    F.col("movie_id").isNull()
    | F.col("genre_id").isNull()
    | F.col("genre_name").isNull()
)

invalid_genre_count = invalid_genres_df.count()
if invalid_genre_count:
    display(invalid_genres_df.limit(20))
    raise ValueError(
        f"Found {invalid_genre_count} genre rows missing a required field"
    )

# Identical movie-genre entries can be collapsed; conflicting names must stop processing.
conflicting_genres_df = (
    silver_genres_source_df
    .groupBy("movie_id", "genre_id")
    .agg(F.countDistinct("genre_name").alias("name_versions"))
    .filter(F.col("name_versions") > 1)
)

conflicting_genre_count = conflicting_genres_df.count()
if conflicting_genre_count:
    display(conflicting_genres_df.limit(20))
    raise ValueError(
        f"Found {conflicting_genre_count} movie-genre keys with conflicting names"
    )

silver_genres_source_df = (
    silver_genres_source_df
    .dropDuplicates(["movie_id", "genre_id"])
    .withColumn(
        "row_hash",
        F.sha2(
            F.to_json(F.struct("movie_id", "genre_id", "genre_name")),
            256,
        ),
    )
    .withColumn("load_timestamp", F.current_timestamp())
)

duplicate_genre_keys = (
    silver_genres_source_df
    .groupBy("movie_id", "genre_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Movies with a present genre collection:", genre_snapshot_candidates_df.count())
print(
    "Movies with an empty genre collection:",
    genre_snapshot_candidates_df.filter(F.col("genre_count") == 0).count(),
)
print(
    "Movies with a missing or unparsed genre collection:",
    parsed_child_movies_df.filter(F.col("genres").isNull()).count(),
)
print("Silver genre rows prepared:", silver_genres_source_df.count())
print("Invalid genre rows:", invalid_genre_count)
print("Conflicting genre keys:", conflicting_genre_count)
print("Duplicate movie-genre keys after preparation:", duplicate_genre_keys)

silver_genres_source_df.printSchema()
display(silver_genres_source_df.limit(10))

In [0]:
from datetime import datetime, timezone
from uuid import uuid4

from pyspark.sql.types import (
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

logs_table = "workspace.tmdb_lakehouse.pipeline_execution_logs"

child_log_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("layer", StringType(), False),
    StructField("load_type", StringType(), False),
    StructField("input_parameter", StringType(), False),
    StructField("batch_id", StringType(), False),
    StructField("start_time", TimestampType(), False),
    StructField("end_time", TimestampType(), False),
    StructField("status", StringType(), False),
    StructField("rows_inserted", LongType(), False),
    StructField("rows_updated", LongType(), False),
    StructField("rows_quarantined", LongType(), False),
    StructField("error_details", StringType(), True),
    StructField("load_timestamp", TimestampType(), False),
])

def write_child_run_log(
    run_id,
    started_at,
    collection_name,
    status,
    rows_inserted,
    rows_updated,
    rows_quarantined=0,
    error_details=None,
):
    finished_at = datetime.now(timezone.utc).replace(tzinfo=None)
    load_type = parsed_child_movies_df.select("load_type").first()["load_type"]

    log_row = [(
        run_id,
        "Bronze-to-Silver",
        load_type,
        f"batch_id={batch_id}; collection={collection_name}",
        batch_id,
        started_at,
        finished_at,
        status,
        rows_inserted,
        rows_updated,
        rows_quarantined,
        error_details,
        finished_at,
    )]

    (
        spark.createDataFrame(log_row, schema=child_log_schema)
        .write.format("delta")
        .mode("append")
        .saveAsTable(logs_table)
    )

print("Child-table audit helper defined.")

In [0]:
from pyspark.sql import functions as F

genres_table = "workspace.tmdb_lakehouse.silver_movie_genres"
state_table = "workspace.tmdb_lakehouse.silver_child_snapshot_state"

genre_run_id = str(uuid4())
genre_run_started_at = datetime.now(timezone.utc).replace(tzinfo=None)
genre_rows_inserted = 0
genre_rows_updated = 0
genre_rows_deleted = 0
genre_rows_quarantined = 0

try:
    state = (
        spark.table(state_table)
        .filter(F.col("collection_name") == "genres")
        .select(
            "movie_id",
            F.col("source_extracted_at").alias("stored_extracted_at"),
            F.col("source_batch_id").alias("stored_batch_id"),
        )
    )

    eligible_genre_snapshots_df = (
        genre_snapshot_candidates_df.alias("incoming")
        .join(state.alias("stored"), on="movie_id", how="left")
        .filter(
            F.col("stored.stored_batch_id").isNull()
            | (
                F.col("incoming.source_extracted_at")
                > F.col("stored.stored_extracted_at")
            )
            | (
                (F.col("incoming.source_extracted_at") == F.col("stored.stored_extracted_at"))
                & (F.col("incoming.source_batch_id") == F.col("stored.stored_batch_id"))
            )
        )
        .select(
            "movie_id",
            "source_extracted_at",
            "source_batch_id",
            "genre_count",
        )
    )

    candidate_count = genre_snapshot_candidates_df.count()
    eligible_count = eligible_genre_snapshots_df.count()

    print("Eligible movie genre snapshots:", eligible_count)

    if eligible_count != candidate_count:
        raise ValueError(
            "Some genre snapshots are older than stored data or have an "
            "ambiguous timestamp. Resolve snapshot ordering before processing."
        )

    eligible_ids_df = eligible_genre_snapshots_df.select("movie_id").distinct()

    eligible_genres_df = silver_genres_source_df.join(
        eligible_ids_df,
        on="movie_id",
        how="inner",
    )

    existing_genres_df = spark.table(genres_table)

    genre_rows_inserted = (
        eligible_genres_df
        .join(existing_genres_df, ["movie_id", "genre_id"], "left_anti")
        .count()
    )

    genre_rows_updated = (
        eligible_genres_df.alias("incoming")
        .join(
            existing_genres_df.alias("stored"),
            (F.col("incoming.movie_id") == F.col("stored.movie_id"))
            & (F.col("incoming.genre_id") == F.col("stored.genre_id")),
            "inner",
        )
        .filter(
            (~F.col("incoming.row_hash").eqNullSafe(F.col("stored.row_hash")))
            | (~F.col("incoming.source_extracted_at").eqNullSafe(F.col("stored.source_extracted_at")))
            | (~F.col("incoming.source_batch_id").eqNullSafe(F.col("stored.source_batch_id")))
            | (~F.col("incoming.source_file").eqNullSafe(F.col("stored.source_file")))
        )
        .count()
    )

    stale_genres_df = (
        existing_genres_df
        .join(eligible_ids_df, on="movie_id", how="inner")
        .join(
            eligible_genres_df.select("movie_id", "genre_id"),
            on=["movie_id", "genre_id"],
            how="left_anti",
        )
        .select("movie_id", "genre_id")
        .dropDuplicates()
    )
    genre_rows_deleted = stale_genres_df.count()

    eligible_genres_df.createOrReplaceTempView("eligible_genres_source")
    stale_genres_df.createOrReplaceTempView("stale_genres_to_delete")

    spark.sql(f"""
        MERGE INTO {genres_table} AS target
        USING eligible_genres_source AS source
        ON target.movie_id = source.movie_id
           AND target.genre_id = source.genre_id

        WHEN MATCHED AND (
            NOT (target.row_hash <=> source.row_hash)
            OR NOT (target.source_extracted_at <=> source.source_extracted_at)
            OR NOT (target.source_batch_id <=> source.source_batch_id)
            OR NOT (target.source_file <=> source.source_file)
        )
        THEN UPDATE SET
            target.genre_name = source.genre_name,
            target.source_batch_id = source.source_batch_id,
            target.source_file = source.source_file,
            target.source_extracted_at = source.source_extracted_at,
            target.row_hash = source.row_hash,
            target.load_timestamp = CASE
                WHEN NOT (target.row_hash <=> source.row_hash)
                THEN source.load_timestamp
                ELSE target.load_timestamp
            END

        WHEN NOT MATCHED THEN INSERT (
            movie_id,
            genre_id,
            genre_name,
            source_batch_id,
            source_file,
            source_extracted_at,
            row_hash,
            load_timestamp
        ) VALUES (
            source.movie_id,
            source.genre_id,
            source.genre_name,
            source.source_batch_id,
            source.source_file,
            source.source_extracted_at,
            source.row_hash,
            source.load_timestamp
        )
    """)

    spark.sql(f"""
        MERGE INTO {genres_table} AS target
        USING stale_genres_to_delete AS stale
        ON target.movie_id = stale.movie_id
           AND target.genre_id = stale.genre_id
        WHEN MATCHED THEN DELETE
    """)

    eligible_genre_snapshots_df.select(
        "movie_id",
        F.lit("genres").alias("collection_name"),
        "source_extracted_at",
        "source_batch_id",
        F.current_timestamp().alias("load_timestamp"),
    ).createOrReplaceTempView("genre_snapshot_state_source")

    spark.sql(f"""
        MERGE INTO {state_table} AS target
        USING genre_snapshot_state_source AS source
        ON target.movie_id = source.movie_id
           AND target.collection_name = source.collection_name

        WHEN MATCHED AND
            source.source_extracted_at > target.source_extracted_at
        THEN UPDATE SET
            target.source_extracted_at = source.source_extracted_at,
            target.source_batch_id = source.source_batch_id,
            target.load_timestamp = source.load_timestamp

        WHEN NOT MATCHED THEN INSERT (
            movie_id,
            collection_name,
            source_extracted_at,
            source_batch_id,
            load_timestamp
        ) VALUES (
            source.movie_id,
            source.collection_name,
            source.source_extracted_at,
            source.source_batch_id,
            source.load_timestamp
        )
    """)

    write_child_run_log(
        genre_run_id,
        genre_run_started_at,
        "genres",
        "SUCCESS",
        genre_rows_inserted,
        genre_rows_updated,
        genre_rows_quarantined,
    )

    print("Genre rows inserted:", genre_rows_inserted)
    print("Genre rows updated:", genre_rows_updated)
    print("Stale genre rows deleted:", genre_rows_deleted)
    print(
        "Genre rows now stored:",
        spark.table(genres_table).count(),
    )

except Exception as exc:
    try:
        write_child_run_log(
            genre_run_id,
            genre_run_started_at,
            "genres",
            "FAILURE",
            genre_rows_inserted,
            genre_rows_updated,
            genre_rows_quarantined,
            str(exc)[:4000],
        )
    except Exception as log_exc:
        raise RuntimeError(
            f"Genre processing failed: {exc}; failure logging also failed: {log_exc}"
        ) from log_exc
    raise

In [0]:
spark.sql("""
SELECT run_id, load_type, batch_id, status,
       rows_inserted, rows_updated, rows_quarantined, error_details
FROM workspace.tmdb_lakehouse.pipeline_execution_logs
WHERE layer = 'Bronze-to-Silver'
  AND input_parameter LIKE '%collection=genres%'
ORDER BY start_time DESC
LIMIT 5
""").show(truncate=False)

In [0]:
from pyspark.sql import functions as F

# Include present empty arrays as snapshot candidates.
studio_snapshot_candidates_df = (
    parsed_child_movies_df
    .filter(F.col("production_companies").isNotNull())
    .select(
        "movie_id",
        "source_extracted_at",
        F.col("batch_id").alias("source_batch_id"),
        F.size("production_companies").alias("studio_count"),
    )
)

duplicate_studio_candidates = (
    studio_snapshot_candidates_df
    .groupBy("movie_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
if duplicate_studio_candidates:
    raise ValueError(
        f"Found {duplicate_studio_candidates} duplicate studio snapshot candidates"
    )

silver_studios_source_df = (
    parsed_child_movies_df
    .filter(F.col("production_companies").isNotNull())
    .select(
        "movie_id",
        "batch_id",
        "source_file",
        "source_extracted_at",
        F.explode("production_companies").alias("studio"),
    )
    .select(
        "movie_id",
        F.col("studio.id").alias("company_id"),
        F.col("studio.name").alias("company_name"),
        F.col("studio.origin_country").alias("origin_country"),
        F.col("batch_id").alias("source_batch_id"),
        "source_file",
        "source_extracted_at",
    )
)

invalid_studios_df = silver_studios_source_df.filter(
    F.col("movie_id").isNull()
    | F.col("company_id").isNull()
    | F.col("company_name").isNull()
)
invalid_studio_count = invalid_studios_df.count()

if invalid_studio_count:
    display(invalid_studios_df.limit(20))
    raise ValueError(
        f"Found {invalid_studio_count} studio rows missing a required field"
    )

# Stop if a movie-company key has conflicting attributes in this batch.
conflicting_studios_df = (
    silver_studios_source_df
    .groupBy("movie_id", "company_id")
    .agg(
        F.countDistinct(
            F.to_json(F.struct("company_name", "origin_country"))
        ).alias("attribute_versions")
    )
    .filter(F.col("attribute_versions") > 1)
)
conflicting_studio_count = conflicting_studios_df.count()

if conflicting_studio_count:
    display(conflicting_studios_df.limit(20))
    raise ValueError(
        f"Found {conflicting_studio_count} movie-company keys with conflicting attributes"
    )

silver_studios_source_df = (
    silver_studios_source_df
    .dropDuplicates(["movie_id", "company_id"])
    .withColumn(
        "row_hash",
        F.sha2(
            F.to_json(
                F.struct(
                    "movie_id",
                    "company_id",
                    "company_name",
                    "origin_country",
                )
            ),
            256,
        ),
    )
    .withColumn("load_timestamp", F.current_timestamp())
)

duplicate_studio_keys = (
    silver_studios_source_df
    .groupBy("movie_id", "company_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Movies with a present studio collection:", studio_snapshot_candidates_df.count())
print(
    "Movies with an empty studio collection:",
    studio_snapshot_candidates_df.filter(F.col("studio_count") == 0).count(),
)
print(
    "Movies with a missing or unparsed studio collection:",
    parsed_child_movies_df.filter(F.col("production_companies").isNull()).count(),
)
print("Silver studio rows prepared:", silver_studios_source_df.count())
print("Invalid studio rows:", invalid_studio_count)
print("Conflicting movie-company keys:", conflicting_studio_count)
print("Duplicate movie-company keys after preparation:", duplicate_studio_keys)

silver_studios_source_df.printSchema()
display(silver_studios_source_df.limit(10))

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.tmdb_lakehouse.silver_movie_studios (
    movie_id BIGINT,
    company_id BIGINT,
    company_name STRING,
    origin_country STRING,
    source_batch_id STRING,
    source_file STRING,
    source_extracted_at TIMESTAMP,
    row_hash STRING,
    load_timestamp TIMESTAMP
)
USING DELTA
""")

spark.sql("""
DESCRIBE TABLE workspace.tmdb_lakehouse.silver_movie_studios
""").show(truncate=False)

In [0]:
from pyspark.sql import functions as F

studios_table = "workspace.tmdb_lakehouse.silver_movie_studios"
state_table = "workspace.tmdb_lakehouse.silver_child_snapshot_state"

studio_run_id = str(uuid4())
studio_run_started_at = datetime.now(timezone.utc).replace(tzinfo=None)
studio_rows_inserted = 0
studio_rows_updated = 0
studio_rows_deleted = 0
studio_rows_quarantined = 0

try:
    stored_studio_state_df = (
        spark.table(state_table)
        .filter(F.col("collection_name") == "studios")
        .select(
            "movie_id",
            F.col("source_extracted_at").alias("stored_extracted_at"),
            F.col("source_batch_id").alias("stored_batch_id"),
        )
    )

    eligible_studio_snapshots_df = (
        studio_snapshot_candidates_df.alias("incoming")
        .join(stored_studio_state_df.alias("stored"), on="movie_id", how="left")
        .filter(
            F.col("stored.stored_batch_id").isNull()
            | (
                F.col("incoming.source_extracted_at")
                > F.col("stored.stored_extracted_at")
            )
            | (
                (F.col("incoming.source_extracted_at") == F.col("stored.stored_extracted_at"))
                & (F.col("incoming.source_batch_id") == F.col("stored.stored_batch_id"))
            )
        )
        .select(
            "movie_id",
            "source_extracted_at",
            "source_batch_id",
            "studio_count",
        )
    )

    candidate_count = studio_snapshot_candidates_df.count()
    eligible_count = eligible_studio_snapshots_df.count()
    print("Eligible movie studio snapshots:", eligible_count)

    if eligible_count != candidate_count:
        raise ValueError(
            "Some studio snapshots are older than stored data, or have an "
            "ambiguous timestamp. Resolve snapshot ordering before processing."
        )

    eligible_ids_df = eligible_studio_snapshots_df.select("movie_id").distinct()

    eligible_studios_df = silver_studios_source_df.join(
        eligible_ids_df,
        on="movie_id",
        how="inner",
    )

    existing_studios_df = spark.table(studios_table)

    studio_rows_inserted = (
        eligible_studios_df
        .join(existing_studios_df, ["movie_id", "company_id"], "left_anti")
        .count()
    )

    studio_rows_updated = (
        eligible_studios_df.alias("incoming")
        .join(
            existing_studios_df.alias("stored"),
            (F.col("incoming.movie_id") == F.col("stored.movie_id"))
            & (F.col("incoming.company_id") == F.col("stored.company_id")),
            "inner",
        )
        .filter(
            (~F.col("incoming.row_hash").eqNullSafe(F.col("stored.row_hash")))
            | (~F.col("incoming.source_extracted_at").eqNullSafe(F.col("stored.source_extracted_at")))
            | (~F.col("incoming.source_batch_id").eqNullSafe(F.col("stored.source_batch_id")))
            | (~F.col("incoming.source_file").eqNullSafe(F.col("stored.source_file")))
        )
        .count()
    )

    stale_studios_df = (
        existing_studios_df
        .join(eligible_ids_df, on="movie_id", how="inner")
        .join(
            eligible_studios_df.select("movie_id", "company_id"),
            on=["movie_id", "company_id"],
            how="left_anti",
        )
        .select("movie_id", "company_id")
        .dropDuplicates()
    )
    studio_rows_deleted = stale_studios_df.count()

    eligible_studios_df.createOrReplaceTempView("eligible_studios_source")
    stale_studios_df.createOrReplaceTempView("stale_studios_to_delete")

    spark.sql(f"""
        MERGE INTO {studios_table} AS target
        USING eligible_studios_source AS source
        ON target.movie_id = source.movie_id
           AND target.company_id = source.company_id

        WHEN MATCHED AND (
            NOT (target.row_hash <=> source.row_hash)
            OR NOT (target.source_extracted_at <=> source.source_extracted_at)
            OR NOT (target.source_batch_id <=> source.source_batch_id)
            OR NOT (target.source_file <=> source.source_file)
        )
        THEN UPDATE SET
            target.company_name = source.company_name,
            target.origin_country = source.origin_country,
            target.source_batch_id = source.source_batch_id,
            target.source_file = source.source_file,
            target.source_extracted_at = source.source_extracted_at,
            target.row_hash = source.row_hash,
            target.load_timestamp = CASE
                WHEN NOT (target.row_hash <=> source.row_hash)
                THEN source.load_timestamp
                ELSE target.load_timestamp
            END

        WHEN NOT MATCHED THEN INSERT (
            movie_id,
            company_id,
            company_name,
            origin_country,
            source_batch_id,
            source_file,
            source_extracted_at,
            row_hash,
            load_timestamp
        ) VALUES (
            source.movie_id,
            source.company_id,
            source.company_name,
            source.origin_country,
            source.source_batch_id,
            source.source_file,
            source.source_extracted_at,
            source.row_hash,
            source.load_timestamp
        )
    """)

    spark.sql(f"""
        MERGE INTO {studios_table} AS target
        USING stale_studios_to_delete AS stale
        ON target.movie_id = stale.movie_id
           AND target.company_id = stale.company_id
        WHEN MATCHED THEN DELETE
    """)

    eligible_studio_snapshots_df.select(
        "movie_id",
        F.lit("studios").alias("collection_name"),
        "source_extracted_at",
        "source_batch_id",
        F.current_timestamp().alias("load_timestamp"),
    ).createOrReplaceTempView("studio_snapshot_state_source")

    spark.sql(f"""
        MERGE INTO {state_table} AS target
        USING studio_snapshot_state_source AS source
        ON target.movie_id = source.movie_id
           AND target.collection_name = source.collection_name

        WHEN MATCHED AND
            source.source_extracted_at > target.source_extracted_at
        THEN UPDATE SET
            target.source_extracted_at = source.source_extracted_at,
            target.source_batch_id = source.source_batch_id,
            target.load_timestamp = source.load_timestamp

        WHEN NOT MATCHED THEN INSERT (
            movie_id,
            collection_name,
            source_extracted_at,
            source_batch_id,
            load_timestamp
        ) VALUES (
            source.movie_id,
            source.collection_name,
            source.source_extracted_at,
            source.source_batch_id,
            source.load_timestamp
        )
    """)

    write_child_run_log(
        studio_run_id,
        studio_run_started_at,
        "studios",
        "SUCCESS",
        studio_rows_inserted,
        studio_rows_updated,
        studio_rows_quarantined,
    )

    print("Studio rows inserted:", studio_rows_inserted)
    print("Studio rows updated:", studio_rows_updated)
    print("Stale studio rows deleted:", studio_rows_deleted)
    print("Studio rows now stored:", spark.table(studios_table).count())

except Exception as exc:
    try:
        write_child_run_log(
            studio_run_id,
            studio_run_started_at,
            "studios",
            "FAILURE",
            studio_rows_inserted,
            studio_rows_updated,
            studio_rows_quarantined,
            str(exc)[:4000],
        )
    except Exception as log_exc:
        raise RuntimeError(
            f"Studio processing failed: {exc}; failure logging also failed: {log_exc}"
        ) from log_exc
    raise

In [0]:
spark.sql("""
SELECT run_id, batch_id, status, rows_inserted, rows_updated,
       rows_quarantined, error_details
FROM workspace.tmdb_lakehouse.pipeline_execution_logs
WHERE layer = 'Bronze-to-Silver'
  AND input_parameter LIKE '%collection=studios%'
ORDER BY start_time DESC
LIMIT 5
""").show(truncate=False)